# Fine-tune CLIP and embed
This notebook is used to fine-tune CLIP with image object - description pairs and to generate embeddings for both modalities.

In [ ]:
import os
import re
import copy
import json
import shutil

import torch
import polars as pl
from PIL import Image
from tqdm import tqdm
import plotly.express as px
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPModel, get_cosine_schedule_with_warmup

INPUT_DATA_PATH = "../../data/fine-tuning_clip/"
STORAGE_PATH = "../../experiments/contrastive_training/"
STORAGE_EMBEDDINGS_PATH = "../../../Open-Grounding-DINO/embeddings_data/"
COLORS = ["#cd968e", "#acb0e0", "#aecbdc", "#bcd5c3", "#bfbfbf"]

### 1. Define PyTorch data loaders

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, csv_file_path, image_dir, processor, max_length=77):
        self.data = pl.read_csv(csv_file_path)
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.data['image_name'][idx])
        image = Image.open(image_path).convert('RGB')
        text = str(self.data['description'][idx])
        
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'raw_text': text
        }

### 2. Define CLIP encoder

In [ ]:
class CLIPEncoder(torch.nn.Module):
    def __init__(self, model, processor, freeze_backbone=False):
        super().__init__()
        self.model = model
        self.processor = processor
        self.freeze_backbone = freeze_backbone
        
        # freeze backbone if requested (only allow training of projection layers)
        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False
            for param in self.model.visual_projection.parameters():
                param.requires_grad = True
            for param in self.model.text_projection.parameters():
                param.requires_grad = True
        # enable gradient computation for all parameters
        else:
            for param in self.model.parameters():
                param.requires_grad = True
    
    def forward(self, input_ids, attention_mask, pixel_values):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_loss=True
        )
        return outputs
    
    def encode_text(self, texts, batch_size=32, max_length=77):
        self.eval()
        all_embeddings = []
        
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                
                inputs = self.processor(
                    text=batch_texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_length
                )
                
                input_ids = inputs['input_ids'].to(next(self.parameters()).device)
                attention_mask = inputs['attention_mask'].to(next(self.parameters()).device)
                
                text_features = self.model.get_text_features(
                    input_ids=input_ids, 
                    attention_mask=attention_mask
                )
                
                text_features = torch.nn.functional.normalize(text_features, p=2, dim=-1)
                all_embeddings.append(text_features.cpu())
        
        return torch.cat(all_embeddings, dim=0)
    
    def encode_images(self, images, batch_size=32):
        self.eval()
        all_embeddings = []
        
        with torch.no_grad():
            for i in range(0, len(images), batch_size):
                batch_images = images[i:i+batch_size]
                
                inputs = self.processor(
                    images=batch_images,
                    return_tensors="pt",
                    padding=True
                )
                
                pixel_values = inputs['pixel_values'].to(next(self.parameters()).device)
                
                image_features = self.model.get_image_features(pixel_values=pixel_values)
                
                image_features = torch.nn.functional.normalize(image_features, p=2, dim=-1)
                all_embeddings.append(image_features.cpu())
        
        return torch.cat(all_embeddings, dim=0)

### 3. Define loss computation

In [ ]:
def compute_loss(clip_encoder, dataloader, device):
    clip_encoder.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            
            outputs = clip_encoder(input_ids, attention_mask, pixel_values)
            loss = outputs.loss
            
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

### 4. Train

#### 4.1. Define hyperparameters, load data and perform initialization

In [ ]:
device = "cuda:0"
experiment_name = "clip_full_1e_6_diff_lr_not_frozen"
max_length = 77
num_epochs = 20
batch_size = 64
learning_rate = 1e-6
weight_decay = 0.01
frozen = False
model_name='openai/clip-vit-base-patch32'    

In [ ]:
try:
    shutil.rmtree(f"{STORAGE_PATH}{experiment_name}")
except:
    pass

os.mkdir(f"{STORAGE_PATH}{experiment_name}")

In [ ]:
# load data and create batches
processor = CLIPProcessor.from_pretrained(model_name)

train_data = ImageTextDataset(f"{INPUT_DATA_PATH}train.csv", INPUT_DATA_PATH, processor, max_length)
val_data = ImageTextDataset(f"{INPUT_DATA_PATH}val.csv", INPUT_DATA_PATH, processor, max_length)
test_data = ImageTextDataset(f"{INPUT_DATA_PATH}test.csv", INPUT_DATA_PATH, processor, max_length)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# initialize CLIP
model = CLIPModel.from_pretrained(model_name)
clip_encoder = CLIPEncoder(model, processor, frozen).to(device)

# define the optimizer and the scheduler for decaying the learning rate
optimizer = torch.optim.AdamW(clip_encoder.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
total_steps = len(train_loader) * num_epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# optimizer = torch.optim.AdamW([
#     {'params': text_encoder.clip_model.parameters(), 'lr': learning_rate / 100},
#     {'params': text_encoder.projector.parameters(), 'lr': learning_rate}
# ], weight_decay=weight_decay)

optimizer = torch.optim.AdamW([
    {'params': clip_encoder.model.text_model.parameters(), 'lr': learning_rate / 10},
    {'params': clip_encoder.model.vision_model.parameters(), 'lr': learning_rate / 10},
    {'params': clip_encoder.model.text_projection.parameters(), 'lr': learning_rate},
    {'params': clip_encoder.model.visual_projection.parameters(), 'lr': learning_rate}
], weight_decay=weight_decay)

#### 4.2. Run the training loop and store results

In [ ]:
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    clip_encoder.train()
    epoch_loss = 0.0
        
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        
        outputs = clip_encoder(input_ids, attention_mask, pixel_values)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_loader))
    val_losses.append(compute_loss(clip_encoder, val_loader, device))
    
    print(f"Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")
    
test_loss = compute_loss(clip_encoder, test_loader, device)
print(f"Test loss: {test_loss:.4f}")

In [ ]:
losses = {
    "epoch": list(range(1, num_epochs + 1)),
    "train_losses": train_losses,
    "val_losses": val_losses,
}
losses_df = pl.from_dict(losses)

fig = px.line(
    losses_df.rename({"train_losses": "train", "val_losses": "val"}),
    x="epoch",
    y=["train", "val"],
    labels={"value": "NT-Xnet loss value"},
    title="Evolution of the contrastive loss value",
    markers=True,
    color_discrete_sequence=COLORS,
)

fig.update_layout(width=1400, height=450)

fig.show()
import plotly.io as pio
pio.write_html(fig, file=f"{STORAGE_PATH}{experiment_name}/loss_evolution.html", auto_open=False)
#fig.write_image(f"{STORAGE_PATH}{experiment_name}/loss_evolution.png", scale=3)

In [ ]:
losses["test_loss"] = test_loss
del losses["epoch"]
results = {
    "max_length": max_length,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "model_name": model_name,
    "frozen": frozen,
    "losses": losses,
}
with open(f"{STORAGE_PATH}{experiment_name}/results.json", "w") as f:
    json.dump(results, f, indent=4)

torch.save(clip_encoder.state_dict(), f"{STORAGE_PATH}{experiment_name}/clip.pth")

### 5. Generate embeddings for descriptions

In [ ]:
clip_encoder = CLIPEncoder(model, processor, frozen).to(device)
clip_encoder.load_state_dict(torch.load(f"{STORAGE_PATH}{experiment_name}/clip.pth"))

In [ ]:
inference_data = pl.read_csv(f"{INPUT_DATA_PATH}test.csv")
inference_data

In [ ]:
text_embeddings = clip_encoder.encode_text(inference_data["description"].to_list(), batch_size, max_length)
image_object_embeddings = clip_encoder.encode_images([Image.open(f"{INPUT_DATA_PATH}{image_name}").convert('RGB') for image_name in inference_data["image_name"].to_list()], batch_size)

In [ ]:
inference_data_updated = inference_data.with_columns(pl.Series("text_embedding_enhanced", text_embeddings)).with_columns(pl.Series("embedding_object_image", image_object_embeddings)).rename({"description": "text"}).to_dicts()

In [ ]:
with open(f"{STORAGE_EMBEDDINGS_PATH}clip_embeddings_test_{experiment_name}.json", "w") as f:
    json.dump(inference_data_updated, f)